# ZeroDefect AI — Google Colab Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sayomphon/ZeroDefect_AI_Industrial_Anomaly/blob/main/notebooks/00_colab_quickstart.ipynb)

This notebook installs the project from `main`, runs the deterministic CPU smoke workflow, displays the generated inspection evidence, and provides an opt-in authenticated Gradio launch. The synthetic result validates pipeline plumbing only; it is not a production model-quality claim.

## 1. Reproducible runtime setup

The cell clones or safely fast-forwards the public repository in a managed Colab runtime, then installs the core and optional demo dependencies. It refuses to overwrite a dirty existing checkout.

In [ ]:
from __future__ import annotations

import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/Sayomphon/ZeroDefect_AI_Industrial_Anomaly.git"
REPOSITORY_BRANCH = "main"
REPOSITORY_NAME = "ZeroDefect_AI_Industrial_Anomaly"
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

git_executable = shutil.which("git")
if IN_COLAB and git_executable is None:
    raise RuntimeError("git is required to prepare the Colab runtime")

if IN_COLAB:
    project_dir = Path("/content") / REPOSITORY_NAME
    if not (project_dir / ".git").is_dir():
        subprocess.run(  # noqa: S603
            [
                git_executable,
                "clone",
                "--depth",
                "1",
                "--branch",
                REPOSITORY_BRANCH,
                REPOSITORY_URL,
                str(project_dir),
            ],
            check=True,
        )
    else:
        dirty = subprocess.run(  # noqa: S603
            [git_executable, "status", "--porcelain"],
            cwd=project_dir,
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()
        if dirty:
            raise RuntimeError("Existing Colab checkout has local changes; refusing to overwrite")
        subprocess.run(  # noqa: S603
            [git_executable, "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
            cwd=project_dir,
            check=True,
        )
else:
    project_dir = Path.cwd().resolve()
    if not (project_dir / "pyproject.toml").is_file():
        raise RuntimeError("Run this notebook from the ZeroDefect AI repository root")

os.chdir(project_dir)
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
if os.environ.get("ZERO_DEFECT_SKIP_INSTALL") != "1":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", ".[demo]"],
        check=True,
    )

print(f"Project directory: {project_dir}")
print(f"Python: {sys.version.split()[0]}")
print(f"Managed Colab runtime: {IN_COLAB}")

## 2. Runtime diagnostics

The statistical baseline is CPU-first. A GPU is not required for this quickstart.

In [ ]:
import platform
import shutil

import numpy as np
from PIL import Image as PILImage

print(f"Platform: {platform.platform()}")
print(f"NumPy: {np.__version__}")
print(f"Pillow: {PILImage.__version__}")
print(f"GPU command available: {shutil.which('nvidia-smi') is not None}")

## 3. End-to-end synthetic smoke test

This executes synthetic dataset generation, normal-only training, threshold calibration on a separate normal-validation split, artifact save/load, MVTec-compatible evaluation, and one overlay prediction.

In [ ]:
import json
import tempfile

from zerodefect_ai.config import ProjectConfig
from zerodefect_ai.workflows import run_smoke

default_output = (
    Path("/content/zerodefect-output")
    if IN_COLAB
    else Path(tempfile.gettempdir()) / "zerodefect-colab-notebook"
)
output_dir = Path(os.environ.get("ZERO_DEFECT_NOTEBOOK_OUTPUT", default_output))
config = ProjectConfig.from_toml(project_dir / "configs" / "base.toml")
report = run_smoke(output_dir, config)

assert report["status"] == "passed", report
assert (output_dir / "smoke_report.json").is_file()
assert (output_dir / "prediction" / "overlay.png").is_file()
print(json.dumps(report, ensure_ascii=False, indent=2))

## 4. Inspect smoke evidence

Perfect synthetic metrics are expected because the generated defect is intentionally easy. Do not use these values as evidence for MVTec AD or factory-camera performance.

In [ ]:
try:
    from IPython.display import Image as NotebookImage
    from IPython.display import display
except ImportError:
    NotebookImage = None
    display = print

image_metrics = report["evaluation"]["image_level"]
metric_summary = {
    "roc_auc": image_metrics["roc_auc"],
    "average_precision": image_metrics["average_precision"],
    "f1": image_metrics["f1"],
    "false_rejects": image_metrics["false_positives"],
    "defect_escapes": image_metrics["false_negatives"],
}
display(metric_summary)
overlay_path = output_dir / "prediction" / "overlay.png"
if NotebookImage is None:
    print(f"Overlay generated: {overlay_path}")
else:
    display(NotebookImage(filename=str(overlay_path)))

## 5. Optional authenticated Gradio UI

A managed Colab runtime cannot expose localhost directly, so Gradio creates a public share URL. The project keeps this disabled by default. Enable it only with non-sensitive synthetic/public images, choose a unique password, and stop the cell when finished. The link is public-facing even though authentication is required.

In [ ]:
import getpass

from app.app import launch_colab_app

ENABLE_PUBLIC_GRADIO = False

if ENABLE_PUBLIC_GRADIO:
    gradio_username = input("Temporary Gradio username: ").strip() or "operator"
    gradio_password = getpass.getpass("Temporary Gradio password (12+ characters): ")
    launch_colab_app(
        output_dir / "artifact",
        username=gradio_username,
        password=gradio_password,
        confirm_public_share=True,
    )
else:
    print("Gradio public share remains disabled. Set ENABLE_PUBLIC_GRADIO=True to opt in.")

## Result

The Colab quickstart is complete when the smoke status is `passed`, the JSON report is present, and the overlay renders. The next quality gate is evaluation on a licensed real dataset such as MVTec AD; that work is intentionally outside this synthetic quickstart.